In [ ]:
# Setup: mount Drive, copy code + data to the fast local disk, decode images once
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
# the Drive folder used for PyramidNet (contains teacher_t45.pt, data.zip, cifar_resnet.py)
hits = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/teacher_t45.pt', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), 'cifar_resnet.py'))]
assert hits, 'Folder with teacher_t45.pt + cifar_resnet.py not found in My Drive'
DRIVE = hits[0]
print('using folder:', DRIVE, '->', sorted(os.listdir(DRIVE)))
assert 'sam_rho' in open(f'{DRIVE}/cifar_resnet.py').read(), 'Replace cifar_resnet.py in the Drive folder with the new (SAM) version'
OUT = f'{DRIVE}/runs'                                # checkpoints/results saved on Drive (survive disconnects)
os.makedirs(OUT, exist_ok=True)
os.chdir('/content')
for f in ['cifar_resnet.py', 'predict.py']:          # always refresh the code
    shutil.copy(f'{DRIVE}/{f}', f)
if not os.path.exists('teacher_t45.pt'):
    shutil.copy(f'{DRIVE}/teacher_t45.pt', 'teacher_t45.pt')
if not os.path.exists('data/train_labels.csv'):
    shutil.unpack_archive(f'{DRIVE}/data.zip', 'data')
WORKERS = min(8, os.cpu_count())
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python cifar_resnet.py --name prepare --prepare_only --data_dir data --cache_dir cache --out_dir prep
# Exactly the round-2 kd45 recipe (Wide-ResNet-28 student, 4.84M, taught by t45) + SAM with rho = 0.05
COMMON = (f'--arch wrn28_s --epochs 250 --teacher_ckpt teacher_t45.pt --sam_rho 0.05 --workers {WORKERS} '
          f'--data_dir data --cache_dir cache --out_dir "{OUT}" --resume')

In [ ]:
# Run 1 (~2-3 h): train on 45k, score on the 5k held-back images -> tells us if SAM beats our best (97.84%)
!python cifar_resnet.py --name kdsam45 --val_split 5000 {COMMON}

In [ ]:
# Held-back score with 10-view TTA. Decision: run the next cell ONLY if this prints > 0.9784
!python predict.py "{OUT}/kdsam45/final.pt" kdsam45_heldout.csv --val --cache_dir cache

In [ ]:
# Run 2 (~2-3 h): same recipe on all 50k images -> the submission model
!python cifar_resnet.py --name kdsam50 {COMMON}
!python predict.py "{OUT}/kdsam50/final.pt" "{DRIVE}/submission_sam_kdsam50.csv" --cache_dir cache

In [ ]:
# Held-back accuracy over training: SAM (kdsam45) vs plain SGD (kd45), same student, same teacher
import json, matplotlib.pyplot as plt
def curve(path):
    h = [r for r in json.load(open(path))['history'] if 'val_acc_ema' in r]
    return [r['epoch'] for r in h], [100 * r['val_acc_ema'] for r in h]
for path, label in [(f'{OUT}/kdsam45/metrics.json', 'SAM'), (f'{DRIVE}/baseline_kd45_metrics.json', 'SGD (round 2)')]:
    print(label, json.load(open(path))['summary'])
plt.figure(figsize=(8, 4.5))
for path, label in [(f'{OUT}/kdsam45/metrics.json', 'KD + SAM (rho 0.05)'),
                    (f'{DRIVE}/baseline_kd45_metrics.json', 'KD + SGD, round 2')]:
    plt.plot(*curve(path), label=label)
plt.ylim(90, 99); plt.xlabel('epoch'); plt.ylabel('held-back accuracy (%)'); plt.grid(alpha=.3); plt.legend()
plt.title('Held-back 5k accuracy (EMA weights), Wide-ResNet-28 student taught by WRN-28-10'); plt.show()

**Pipeline.** The student is the round-2 pre-activation Wide-ResNet-28 (widths 64/128/224, 4,839,578 trainable parameters), trained from scratch for 250 epochs on random crop + flip + TrivialAugment + Cutout with label smoothing 0.1 and a weight EMA, and taught by the 36M-parameter WRN-28-10 teacher (KD loss: 0.9·KL at T = 4 + 0.1·CE; the teacher is never submitted). The only change from round 2 is the optimizer step: Sharpness-Aware Minimization (SAM, ρ = 0.05) around SGD + Nesterov (lr 0.1, 5-epoch warmup, cosine decay, weight decay 5e-4, batch 128, fp16). Each step computes the gradient, moves the weights by ρ·g/‖g‖ towards higher loss, recomputes the gradient there, returns to the original weights and applies that second gradient.

**Why.** The test images are new photos with almost the same brightness, contrast and sharpness as the training set, so the score is limited by how well the model generalises rather than by a low-level shift. SAM steers training towards flat minima, where the loss stays low under small weight changes, which is linked to better generalisation (Foret et al. 2021). Changing only the optimizer makes `kdsam45` directly comparable to kd45 on the same 5k held-back split: run `kdsam50` and submit `submission_sam_kdsam50.csv` only if kdsam45 beats 97.84% (10-view TTA). If Colab disconnects, re-run cell 1 and the interrupted cell: `--resume` continues from the last checkpoint on Drive (saved every 10 epochs).